In [ ]:
!pip install -U  langchain-text-splitters langchain-huggingface langchain-community faiss-cpu

In [ ]:
!pip install langchain langchain-community langchain-huggingface
!pip install chromadb pypdf sentence-transformers

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = "KEEP YOUR OWN TOKEN HERE"

In [ ]:
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS,Chroma
from langchain_core.prompts import PromptTemplate
from huggingface_hub import InferenceClient
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_9763/1402145861.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS,Chroma


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("Upload a PDF.")

resume_filename = next(iter(uploaded.keys()))

if not resume_filename.lower().endswith(".pdf"):
    raise ValueError("file  must be a PDF.")

Saving cricket_rag_practice.pdf to cricket_rag_practice.pdf


In [ ]:
pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)

Uploaded file: cricket_rag_practice.pdf


In [ ]:
loader=PyPDFLoader(pdf_path)
docs=loader.load()

##Chunking the loaded Document


In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

In [ ]:
chunks=splitter.split_documents(docs)

In [ ]:
len(chunks)

9

In [ ]:
chunks[0].page_content

'Cricket Knowledge Handbook\nA practice document for learning document loading, chunking, embeddings, Chroma vector stores,\nsimilarity search, and RAG.\n1. Introduction to Cricket\nCricket is a bat-and-ball team sport played between two teams. A team normally has eleven players. The\nmajor forms of the game are Test cricket, One Day Internationals (ODIs), and Twenty20 (T20) cricket. The\nformats differ mainly in match duration, number of overs, and tactical priorities.\nA cricket match is organized around innings. During an innings, one team bats while the other team bowls\nand fields. The batting team attempts to score runs, while the bowling and fielding team attempts to restrict\nscoring and dismiss batters.\n2. Basic Cricket Terminology\n Term\nMeaning\nInnings\nA period in which a team or batter is given the opportunity to score runs.\nOver\nA set of six legal deliveries bowled by one bowler.\nRun\nA unit of scoring obtained by the batting side.\nBoundary\nA shot reaching the bou

##Converting the chunks to Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector_store=FAISS.from_documents(chunks,embedding_model)

#starting Reteiver Process

In [ ]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)


In [ ]:
question="Who is the most famous batsman"

respond=retriever.invoke(question)

In [ ]:
context="".join (chunk.page_content for chunk in respond)

In [ ]:
context

'11. Famous Indian Cricket Players\n Player\nRole / commonly associated strengths\nSachin Tendulkar\nTop-order batter; renowned for extensive international run scoring.\nVirat Kohli\nTop-order batter; known for consistency, chasing ability, and fitness.\nRohit Sharma\nTop-order batter and former captain; known for timing and large scores.\nMS Dhoni\nWicketkeeper-batter and former captain; known for finishing and leadership.\nJasprit Bumrah\nFast bowler; known for yorkers, variations, and death-over bowling.\nRavindra Jadeja\nAll-rounder; contributes with batting, left-arm spin, and fielding.\nRavichandran Ashwin\nSpin bowler and all-round contributor; known for tactical variation.\n12. IPL Team Examples\nThe Indian Premier League is a franchise-based T20 competition. Teams use a combination of domestic\nand international players and construct squads around specialized roles. Common strategic roles include\nopening batters, finishers, pace bowlers, spinners, wicketkeepers, and all-round

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Answer ONLY from the provided PDF context.
If the context is insufficient, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

#generation

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint ,ChatHuggingFace

In [ ]:
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text_generation",
     huggingfacehub_api_token=os.environ["HF_TOKEN"]
)


In [ ]:
model=ChatHuggingFace(llm=llm)

In [ ]:
final_prompt=prompt.invoke({'context':context,'question':question})

In [ ]:
response=model.invoke(final_prompt)

In [ ]:
response.content

'Sachin Tendulkar.'

#Making of  Chains

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parser=StrOutputParser()

In [ ]:
def retrieved_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
print(retrieved_docs(respond))

11. Famous Indian Cricket Players
 Player
Role / commonly associated strengths
Sachin Tendulkar
Top-order batter; renowned for extensive international run scoring.
Virat Kohli
Top-order batter; known for consistency, chasing ability, and fitness.
Rohit Sharma
Top-order batter and former captain; known for timing and large scores.
MS Dhoni
Wicketkeeper-batter and former captain; known for finishing and leadership.
Jasprit Bumrah
Fast bowler; known for yorkers, variations, and death-over bowling.
Ravindra Jadeja
All-rounder; contributes with batting, left-arm spin, and fielding.
Ravichandran Ashwin
Spin bowler and all-round contributor; known for tactical variation.
12. IPL Team Examples
The Indian Premier League is a franchise-based T20 competition. Teams use a combination of domestic
and international players and construct squads around specialized roles. Common strategic roles include
opening batters, finishers, pace bowlers, spinners, wicketkeepers, and all-rounders.
Royal Challenger

In [ ]:
parallel_chain = RunnableParallel(
    {
        "context": retriever | RunnableLambda(retrieved_docs),
        "question": RunnablePassthrough()
    }
)

In [ ]:
parallel_chain.invoke("who is god of cricket")

{'context': '11. Famous Indian Cricket Players\n Player\nRole / commonly associated strengths\nSachin Tendulkar\nTop-order batter; renowned for extensive international run scoring.\nVirat Kohli\nTop-order batter; known for consistency, chasing ability, and fitness.\nRohit Sharma\nTop-order batter and former captain; known for timing and large scores.\nMS Dhoni\nWicketkeeper-batter and former captain; known for finishing and leadership.\nJasprit Bumrah\nFast bowler; known for yorkers, variations, and death-over bowling.\nRavindra Jadeja\nAll-rounder; contributes with batting, left-arm spin, and fielding.\nRavichandran Ashwin\nSpin bowler and all-round contributor; known for tactical variation.\n12. IPL Team Examples\nThe Indian Premier League is a franchise-based T20 competition. Teams use a combination of domestic\nand international players and construct squads around specialized roles. Common strategic roles include\nopening batters, finishers, pace bowlers, spinners, wicketkeepers, a

In [ ]:
rag_chain = (
    parallel_chain
    | prompt
    | model
    | parser
)

In [ ]:
question="Which player is described as a fast bowler?"

In [ ]:
answer=rag_chain.invoke(question)

In [ ]:
print(parser.invoke(answer))

Jasprit Bumrah.


In [ ]:
questions = [
    "What are the three major formats of cricket?",
    "What is a maiden over?",
    "Which player is described as a fast bowler?",
    "Which players are described as all-rounders?",
    "What does bowling economy rate measure?",
    "What are the common bowling roles in T20 cricket?",
    "What is the difference between a fast bowler and a spin bowler?",
    "Which player is associated with Mumbai Indians, and what type of bowler is he?",
    "Who won the 2025 Cricket World Cup?",
    "What is Virat Kohli's highest ODI score?"
]

for question in questions:
    print("\nQUESTION:", question)
    print("ANSWER:", rag_chain.invoke(question))
    print("-" * 80)


QUESTION: What are the three major formats of cricket?
ANSWER: The three major formats of cricket are Test cricket, One Day Internationals (ODIs), and Twenty20 (T20) cricket.
--------------------------------------------------------------------------------

QUESTION: What is a maiden over?
ANSWER: A maiden over is an over in which the bowler concedes no runs from the bat or extras, subject to the laws of the game.
--------------------------------------------------------------------------------

QUESTION: Which player is described as a fast bowler?
ANSWER: Jasprit Bumrah.
--------------------------------------------------------------------------------

QUESTION: Which players are described as all-rounders?


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6abbc92c-61082d212af4bd864ccd549c;65555913-366c-46e6-b95b-68546e596120)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [ ]:
print(pdf_path)

cricket_rag_practice.pdf
